# Práctico 2: Manipulación de tablas y series de tiempo 

## Obtención de los datos

In [1]:
import pandas as pd
df=pd.read_csv("starbucks.csv")

### Primeros datos

In [2]:
df.head(10) #Impresión de los primeros 10 registros

,Store Number,Store Name,Address,City
0,10429-100710,Palmdale & Hwy 395,14136 US Hwy 395 Adelanto CA,Adelanto
1,635-352,Kanan & Thousand Oaks,5827 Kanan Road Agoura CA,Agoura
2,74510-27669,Vons-Agoura Hills #2001,5671 Kanan Rd. Agoura Hills CA,Agoura Hills
3,29839-255026,Target Anaheim T-0677,8148 E SANTA ANA CANYON ROAD AHAHEIM CA,AHAHEIM
4,23463-230284,Safeway - Alameda 3281,2600 5th Street Alameda CA,Alameda
5,6479-62999,Park & Central Alameda,1364 Park Street Alameda CA,Alameda
6,5535-728,Webster & Atlantic - Alameda,720 Atlantic Avenue Alameda CA,Alameda
7,74877-100291,Safeway - Alameda #2708,2227 South Shore Center Alameda CA,Alameda
8,11161-103516,Tilden & Blanding,"2671 Blanding Avene, D Alameda CA",Alameda
9,19859-196187,Target Alameda T-2829,2700 Fifth St Alameda CA,Alameda


## Operaciones con tablas

### a. Búsqueda de tienda por dirección utilizando .loc

In [3]:
df.loc[df['Address']== '1444 Shattuck Place Berkeley CA'] #Se busca en la columna Address el registro que contenga la dirección

,Store Number,Store Name,Address,City
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley


### b. Filtrado de registros por ciudad (Berkeley)

In [4]:
df.loc[df['City']== 'Berkeley'] #Se busca en la columna City los registros que contengan a Berkeley como ciudad

,Store Number,Store Name,Address,City
153,5406-945,2224 Shattuck - Berkeley,2224 Shattuck Avenue Berkeley CA,Berkeley
154,570-512,Solano Ave,1799 Solano Avenue Berkeley CA,Berkeley
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley
156,19864-202264,Telegraph & Ashby,3001 Telegraph Avenue Berkeley CA,Berkeley
157,9217-9253,2128 Oxford St.,2128 Oxford Street Berkeley CA,Berkeley


### c. Obtención de coordenadas geográficas (latitud y longitud)


In [5]:
from geopy.geocoders import Nominatim #Convertir direcciones en coordenadas
geolocator = Nominatim(user_agent="kaggle_learn") #Crea el geolocalizador que hace la busqueda
direccionTienda= df['Address'][14] #Prueba con la fila 15
locationPrueba = geolocator.geocode(direccionTienda) #Toma la dirección y devuelve un objeto(dirección, latitud, longitud)
print(locationPrueba.address)
print((locationPrueba.latitude, locationPrueba.longitude))

1220, West Main Street, Alhambra, Los Angeles County, California, 91801, United States
(34.091476, -118.1361991)


### d. Geocodificación y manejo de errores

In [6]:
import time
erroresDireccion= []

limite = 100  # límite de consultas
contador = 0

#Obtener latitud y longitud, se usa geolocalizador anterior

def obtener_coordenadas(direccion):
    global contador
    
    if contador >= limite:  # límite para no gastar consultas
        return None, None
    
    try:
        location=geolocator.geocode(direccion) #Toma la dirección y devuelve un objeto(dirección, latitud, longitud)
        time.sleep(1)  # se aumenta el tiempo para evitar saturar el servicio
        contador += 1

        if location: #Si encuentra la dirección devuelve latitud y longitud
            return location.latitude, location.longitude
        else:
            erroresDireccion.append(direccion) # Ocurre error, se agrega a lista de errores de dirección
            return None, None 
    except:
        erroresDireccion.append(direccion)
        contador += 1
        return None, None
    
        
df[['Latitude', 'Longitude']] = df['Address'].apply( lambda x: pd.Series(obtener_coordenadas(x))) #Para la columna address se crean dos nuevas columnas (Latitud y Longitud) y para cada fila se llama a la función

#Mostramos resultados
print(df.head())
print("Direcciones con error:", erroresDireccion)


   Store Number               Store Name  \
0  10429-100710       Palmdale & Hwy 395   
1       635-352    Kanan & Thousand Oaks   
2   74510-27669  Vons-Agoura Hills #2001   
3  29839-255026    Target Anaheim T-0677   
4  23463-230284   Safeway - Alameda 3281   

                                   Address          City   Latitude  \
0             14136 US Hwy 395 Adelanto CA      Adelanto  34.506998   
1                5827 Kanan Road Agoura CA        Agoura  34.155889   
2           5671 Kanan Rd. Agoura Hills CA  Agoura Hills  34.153550   
3  8148 E SANTA ANA CANYON ROAD AHAHEIM CA       AHAHEIM        NaN   
4               2600 5th Street Alameda CA       Alameda  37.785531   

    Longitude  
0 -117.399952  
1 -118.756569  
2 -118.759190  
3         NaN  
4 -122.280575  
Direcciones con error: ['8148 E SANTA ANA CANYON ROAD AHAHEIM CA', '2671 Blanding Avene, D Alameda CA', '2210-J South Shore Drive Alameda CA', '2400 W. Commonwealth Alhambra CA', '26932 La Paz Ave Aliso Viejo CA'

### e. Corrección de direcciones con fallos de geocodificación